# step00b - QC/QA: Filter and Loop Edit before/after

Diagnostic comparison plots for two SBE pre-processing modules, on real casts:

* **Filter (pressure)** - pressure and its rate of change dP/dt, before vs after the low-pass
  filter. Shows whether filtering meaningfully reduces pressure noise ("dithering") on this
  instrument. (Leah McRaven / Ella Cedarholm training, filter section.)
* **Loop Edit** - the cast with vs without loop-edit flagging, for both methods
  (percent-of-mean-speed, the calm/default; and fixed-minimum-velocity, the rough-weather one).
  Shows which scans each method removes on slowdowns/reversals.

This is a QA tool, not a processing step: it does not change the L1/L2 products. It produces
figures you inspect (and bring to the CTD processing meeting) to choose filter/loop-edit settings.

> **Instrument note:** this CTD is an SBE 19plus V2 (strain-gauge pressure, single T/C), not the
> SBE 9plus (Digiquartz) in the training deck. The pressure-dithering that Filter removes is a
> Digiquartz phenomenon, so on a strain-gauge sensor Filter may do *less*. That is exactly what the
> before/after plot is here to reveal - do not assume the 9plus result transfers.

**Status:** the plotting engine is complete and runs on any pair of `.cnv` files. The cells that
*run* FilterW.exe / LoopEditW.exe to generate those pairs are marked `# SLOT` and are filled in once
the Filter and two Loop Edit PSAs exist. Until then, point the loaders at cnv files you generate.


## 1. Imports and settings

In [ ]:
from __future__ import annotations
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except Exception:
    def display(obj): print(obj)


In [ ]:
# ===========================================================================
# Paths / settings
# ===========================================================================
CTD_ROOT   = Path(r"C:\Projects\ctd_pipeline")
CRUISE_ID  = "P45_06"
CRUISE_DIR = CTD_ROOT / "cruises" / CRUISE_ID

# Which casts to inspect. Pick one calm and one rougher cast if you know them.
TEST_CASTS = ["P45_06_CTD_03", "P45_06_CTD_06"]

# Output folder for the QA figures.
QC_PLOT_DIR = CRUISE_DIR / "L2" / "CTD" / "qc_plots"
QC_PLOT_DIR.mkdir(parents=True, exist_ok=True)

# SBE binaries + PSAs (used by the SLOT cells that run the modules).
SBE_BIN_DIR = Path(r"C:\Program Files (x86)\Sea-Bird\SBEDataProcessing-Win32")
PSA_DIR     = CTD_ROOT / "psa"
FILTER_PSA          = PSA_DIR / "05_filter.psa"
LOOPEDIT_PCTMEAN_PSA= PSA_DIR / "06a_loopedit_percentmean.psa"
LOOPEDIT_FIXEDMIN_PSA=PSA_DIR / "06b_loopedit_fixedmin.psa"

# Column names (SBE 19plus V2, from real files).
COL_PRESSURE = "prdM"     # or prDM
COL_TIME_S   = "timeS"    # elapsed seconds
COL_FLAG     = "flag"     # loop edit badflag (0 = good, non-zero/negative = flagged)
COL_TEMP     = "tv290C"
COL_SAL      = "sal00"
BADFLAG_VALUE = -9.990e-29  # SBE marks flagged scans with this in data columns

FIG_DPI = 200
print("Cruise:", CRUISE_ID, "| casts:", TEST_CASTS)
print("QA plots ->", QC_PLOT_DIR)


## 2. CNV reader (keeps flagged scans, unlike the science readers)

In [ ]:
def parse_cnv_raw(path: Path) -> pd.DataFrame:
    """Parse a .cnv to a numeric DataFrame, KEEPING all scans (incl. loop-edit flagged).
    The science readers drop flagged rows; here we need them to show what Loop Edit removed."""
    text = path.read_text(encoding="latin-1", errors="replace").splitlines()
    names, data_start = {}, None
    nr = re.compile(r"#\s*name\s+(\d+)\s*=\s*([^:]+?)\s*[:=]", re.I)
    for i, line in enumerate(text):
        s = line.strip()
        if s.startswith("#") or s.startswith("*"):
            m = nr.search(line)
            if m: names[int(m.group(1))] = m.group(2).strip()
            if s.upper().startswith("*END*"): data_start = i + 1; break
    if data_start is None:
        raise ValueError(f"No *END* in {path.name}")
    cols = [names[k] for k in sorted(names)]
    rows = [ln.split()[:len(cols)] for ln in text[data_start:]
            if ln.strip() and len(ln.split()) >= len(cols)]
    df = pd.DataFrame(rows, columns=cols).apply(pd.to_numeric, errors="coerce")
    return df


def pick(df, *cands):
    for c in cands:
        if c in df.columns: return c
    # loose match
    for c in df.columns:
        if any(x.lower() in c.lower() for x in cands): return c
    return None


## 3. Filter before/after plot

Two panels: (left) pressure vs scan, raw vs filtered, overlaid; (right) dP/dt (package velocity
proxy) vs scan, raw vs filtered. Filtering should smooth the dP/dt trace - if raw and filtered
look identical, filtering is doing little on this instrument (a real, reportable finding).

In [ ]:
def plot_filter_before_after(cast_id, df_raw, df_filt):
    pr = pick(df_raw, COL_PRESSURE, "prDM", "prM", "pressure")
    ts = pick(df_raw, COL_TIME_S, "timeS", "scan")
    if pr is None:
        print(f"{cast_id}: no pressure column"); return None
    x_raw  = df_raw[ts] if ts else np.arange(len(df_raw))
    x_filt = df_filt[ts] if ts and ts in df_filt else np.arange(len(df_filt))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
    ax1.plot(x_raw,  df_raw[pr],  lw=0.6, color="#bbbbbb", label="raw")
    ax1.plot(x_filt, df_filt[pr], lw=0.9, color="#1f77b4", label="filtered")
    ax1.set_xlabel("elapsed time (s)" if ts else "scan"); ax1.set_ylabel("pressure (db)")
    ax1.set_title(f"{cast_id}: pressure"); ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

    dpr_raw  = np.gradient(pd.to_numeric(df_raw[pr], errors="coerce").to_numpy(float))
    dpr_filt = np.gradient(pd.to_numeric(df_filt[pr], errors="coerce").to_numpy(float))
    ax2.plot(x_raw,  dpr_raw,  lw=0.5, color="#bbbbbb", label="raw dP/dt")
    ax2.plot(x_filt, dpr_filt, lw=0.8, color="#d62728", label="filtered dP/dt")
    ax2.set_xlabel("elapsed time (s)" if ts else "scan"); ax2.set_ylabel("dP/dt (db/scan)")
    ax2.set_title("rate of change (dithering shows here)"); ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

    # quick numeric summary of the effect
    rms_raw  = float(np.nanstd(dpr_raw)); rms_filt = float(np.nanstd(dpr_filt))
    fig.suptitle(f"Filter effect on dP/dt scatter: std {rms_raw:.3f} -> {rms_filt:.3f} db/scan "
                 f"({100*(1-rms_filt/rms_raw):.0f}% reduction)" if rms_raw else "", fontsize=9, y=1.02)
    out = QC_PLOT_DIR / f"{cast_id}__filter_before_after.png"
    fig.savefig(out, dpi=FIG_DPI, bbox_inches="tight"); plt.close(fig)
    return out, rms_raw, rms_filt


## 4. Loop Edit before/after plot

Loop Edit sets a badflag on scans during slowdowns/reversals (it does not delete them). This panel
plots pressure vs scan and marks the flagged scans, so you see exactly what each method removes.
Run it for both methods to compare percent-of-mean (calm) vs fixed-min-velocity (rough).

In [ ]:
def flagged_mask(df):
    """True where Loop Edit flagged the scan. SBE writes a badflag column, or marks data columns
    with the badflag sentinel value; handle both."""
    if COL_FLAG in df.columns:
        f = pd.to_numeric(df[COL_FLAG], errors="coerce").fillna(0)
        return f != 0
    # fallback: a science column full of the sentinel
    for c in (COL_TEMP, COL_SAL, COL_PRESSURE):
        if c in df.columns:
            return np.isclose(pd.to_numeric(df[c], errors="coerce"), BADFLAG_VALUE, atol=1e-30)
    return pd.Series(False, index=df.index)


def plot_loopedit_before_after(cast_id, df_edited, method_label):
    pr = pick(df_edited, COL_PRESSURE, "prDM", "prM")
    ts = pick(df_edited, COL_TIME_S, "timeS", "scan")
    if pr is None:
        print(f"{cast_id}: no pressure column"); return None
    x = df_edited[ts] if ts else np.arange(len(df_edited))
    flg = flagged_mask(df_edited)

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(x, df_edited[pr], lw=0.6, color="#1f77b4", label="kept scans")
    if flg.any():
        ax.scatter(x[flg], df_edited[pr][flg], s=8, color="#d62728", zorder=3,
                   label=f"flagged ({int(flg.sum())} scans, {100*flg.mean():.1f}%)")
    ax.invert_yaxis()
    ax.set_xlabel("elapsed time (s)" if ts else "scan"); ax.set_ylabel("pressure (db)")
    ax.set_title(f"{cast_id}: Loop Edit - {method_label}")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    slug = re.sub(r"[^a-z0-9]+", "_", method_label.lower()).strip("_")
    out = QC_PLOT_DIR / f"{cast_id}__loopedit_{slug}.png"
    fig.savefig(out, dpi=FIG_DPI, bbox_inches="tight"); plt.close(fig)
    return out, int(flg.sum()), float(flg.mean())


## 5. SLOT - run the modules to generate before/after cnv pairs

**These cells are filled in once the three PSAs exist** (`05_filter.psa`,
`06a_loopedit_percentmean.psa`, `06b_loopedit_fixedmin.psa`). Each runs the SBE binary on a test
cast's aligned cnv (`_al.cnv` from step01's L2/CTD/Align_CTD) and writes a comparison cnv into the
work area. The command pattern mirrors step01: `EXE.exe /pPSA /iINPUT /oOUTDIR /fSTEM /s`.

Until then, set the paths below to cnv files you already have (e.g. an unfiltered vs a
GUI-filtered cnv) so the plotting cells in Section 6 can run.

In [ ]:
import subprocess

def run_sbe(exe_name, psa, in_cnv, out_dir, out_stem):
    """Run one SBE module in-place for the QC comparison. Mirrors step01's command construction.
    Returns the output .cnv path (or None on failure)."""
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [str(SBE_BIN_DIR / exe_name),
           f"/p{psa}", f"/i{in_cnv}", f"/o{out_dir}", f"/f{out_stem}", "/s"]
    print("   ", " ".join(str(x) for x in cmd))
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        print("    WARNING return code", res.returncode, (res.stderr or "")[:200])
    out = out_dir / f"{out_stem}.cnv"
    return out if out.exists() else None


WORK = CTD_ROOT / "_sbe_work" / "qc"
WORK.mkdir(parents=True, exist_ok=True)
# Input to Filter: the aligned cnv from step01 (L2/CTD/Align_CTD/<cast>_al.cnv).
# Leah's order filters pressure early; here we filter the aligned cnv so the QC plot isolates the
# filter's effect on this instrument. (For the meeting we are demonstrating the module, not locking
# the final pipeline order - that is the recipe decision still open with Leah.)
ALIGN_DIR = CRUISE_DIR / "L2" / "CTD" / "Align_CTD"

def qc_process_cast(cast):
    """Produce the before/after cnv set for one cast. Returns dict of DataFrames (or Nones)."""
    aligned = ALIGN_DIR / f"{cast}_al.cnv"
    if not aligned.exists():
        print(f"  {cast}: no aligned cnv at {aligned} - run step01 first."); return {}
    out = {"aligned": parse_cnv_raw(aligned)}

    # SLOT A: Filter (pressure only, 1.0 s) on the aligned cnv
    filt = run_sbe("FilterW.exe", FILTER_PSA, aligned, WORK, f"{cast}_filt")
    out["filtered"] = parse_cnv_raw(filt) if filt else None

    # SLOT B: Loop Edit both methods, on the FILTERED cnv (Leah: filter pressure before loop edit)
    src = filt if filt else aligned
    le_pct = run_sbe("LoopEditW.exe", LOOPEDIT_PCTMEAN_PSA, src, WORK, f"{cast}_le_pct")
    le_fix = run_sbe("LoopEditW.exe", LOOPEDIT_FIXEDMIN_PSA, src, WORK, f"{cast}_le_fix")
    out["loopedit_pct"] = parse_cnv_raw(le_pct) if le_pct else None
    out["loopedit_fix"] = parse_cnv_raw(le_fix) if le_fix else None
    return out

print("Module-running wired. qc_process_cast(cast) runs Filter + both Loop Edits on a cast.")
print("Requires: PSAs present, step01 Align_CTD output present, SBE binaries at SBE_BIN_DIR.")


## 6. Run the comparisons and collect a summary

In [ ]:
summary = []

for cast in TEST_CASTS:
    print(f"\n=== {cast} ===")
    dfs = qc_process_cast(cast)
    if not dfs:
        continue

    # ---- Filter before/after ----
    if dfs.get("aligned") is not None and dfs.get("filtered") is not None:
        r = plot_filter_before_after(cast, dfs["aligned"], dfs["filtered"])
        if r:
            summary.append({"cast": cast, "check": "filter",
                            "dpdt_std_raw": round(r[1], 4), "dpdt_std_filt": round(r[2], 4),
                            "pct_reduction": round(100 * (1 - r[2] / r[1]), 1) if r[1] else None})
            print(f"  filter: dP/dt std {r[1]:.3f} -> {r[2]:.3f}")

    # ---- Loop Edit: both methods ----
    for key, label in [("loopedit_pct", "percent-of-mean"),
                       ("loopedit_fix", "fixed-min-velocity")]:
        if dfs.get(key) is not None:
            r = plot_loopedit_before_after(cast, dfs[key], label)
            if r:
                summary.append({"cast": cast, "check": f"loopedit_{label}",
                                "n_flagged": r[1], "pct_flagged": round(100 * r[2], 2)})
                print(f"  loopedit {label}: {r[1]} scans flagged ({100*r[2]:.1f}%)")

if summary:
    sdf = pd.DataFrame(summary)
    display(sdf)
    sdf.to_csv(QC_PLOT_DIR / "qc_filter_loopedit_summary.csv", index=False, encoding="utf-8-sig")
    print(f"\nSummary + figures -> {QC_PLOT_DIR}")
else:
    print("\nNo comparisons produced. Check that step01 has run (Align_CTD cnv present) and the")
    print("three PSAs exist in the psa/ folder.")
